# Lab 10 — Deep Reinforcement Learning on `highway-env`

**ELEC5308: Intelligent Information Engineering Practice**


> **This notebook is fully self-contained.** Every helper function (environment setup, training
> wrappers, evaluation, GIF/MP4 recording, plotting, results I/O) is defined directly in the cells
> below — there is no separate Python package to install or path to configure. The **only** install
> you need is the install cell at the top, for the external RL libraries themselves
> (`gymnasium`, `highway-env`, `stable-baselines3`, `torch`).


> ⚠️ **Mandatory task time budget: 30 minutes.** The "Mandatory Task" section (Steps 2–5 below) is
> designed to fit that budget on a CPU laptop. "Optional Extensions" at the end is **ungraded**
> take-home material — do not start it until the mandatory task's artefacts are saved.

> ✍️ **AI Journal reminder:** log any significant Copilot/ChatGPT/Claude interaction you use while
> completing this notebook, per the course AI Usage Policy.


## Step 0 — Install and import

**Install just once per machine/Colab session** — if you re-run this notebook later in the same
session, skip straight to the imports cell.

In [1]:
import importlib.util
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except (ModuleNotFoundError, ValueError):
    IN_COLAB = False

# Keep the lab's required package set.  Use the active Python interpreter so the
# packages are installed into the same environment as this notebook kernel.
REQUIRED_PACKAGES = [
    "gymnasium==1.3.0",
    "highway-env",
    "stable-baselines3",
]

if not IN_COLAB:
    REQUIRED_PACKAGES += [
        "torch",
        "opencv-python-headless",
        "pillow",
        "matplotlib",
        "numpy",
        "pandas",
    ]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--progress-bar", "on", *REQUIRED_PACKAGES],
    check=True,
)

# Do not force a second Torch wheel installation: the normal PyPI Torch package
# is sufficient for this CPU lab and avoids version conflicts.
print("Install complete.")

Install complete.


In [2]:
import sys
import os
import json
import time
import platform
import random

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg" if IN_COLAB else matplotlib.get_backend())
import matplotlib.pyplot as plt
from PIL import Image

import gymnasium as gym
import highway_env  # Registers highway-env environments with Gymnasium.

import torch
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback, CallbackList

print(f"Python: {platform.python_version()}")
print(f"gymnasium: {gym.__version__}")
print(f"torch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
print(f"stable-baselines3: imported successfully")
print(f"highway-fast-v0 available: {'highway-fast-v0' in gym.registry}")

Python: 3.13.16
gymnasium: 1.3.0
torch: 2.11.0+cpu | CUDA available: False
stable-baselines3: imported successfully
highway-fast-v0 available: True


---
## Library code (run once, top to bottom)

The cells below are the "library" living directly in this notebook so there is nothing to install or
import-path-configure. **Run every cell in this section once, in order**, before jumping to
"Mandatory Task" below. You only need to **edit** the one cell marked `TODO`.


### `diagnostics` — secondary-signal metrics (MANDATORY — implement this)

Lecture reference: *"The Golden Rule of Diagnostics"* (slide 51) and *"Reading a Training Curve"*
(slide 52):

> "Never read reward in isolation. Always pair raw reward with a secondary signal."

An "episode record" is a `dict` with the keys produced by the `rollout(...)` function defined later
in this notebook:

```
reward   : float  -- total return for the episode
length   : int    -- number of environment steps taken before termination
crashed  : bool   -- True if the episode ended in a collision (highway-env's info["crashed"])
```

Implement the three functions below so that the **self-check cell right after this one** prints
`ALL CHECKS PASSED`. Do not change the function signatures or remove the
`AUTOGRADER_EXTRACT_START` / `AUTOGRADER_EXTRACT_END` comment markers — the autograder locates this
exact cell by those markers to run its own hidden test suite against your implementation.


In [3]:
# === AUTOGRADER_EXTRACT_START: diagnostics ===

def collision_rate(episodes):
    """Return the fraction of episodes in which the agent crashed.

    Parameters
    ----------
    episodes : sequence of dict
        Each dict has a boolean "crashed" field.

    Returns
    -------
    float in [0.0, 1.0]. Return 0.0 for an empty sequence (do not raise or return NaN).
    """
    if not episodes:
        return 0.0
    return sum(bool(e["crashed"]) for e in episodes) / len(episodes)


def success_rate(episodes):
    """Return the fraction of episodes that ended WITHOUT a collision.

    This is the complement of collision_rate, but implement it directly (do not just
    return 1 - collision_rate(...) -- the hidden autograder tests check both functions
    separately so a bug in one is not masked by the other).

    Returns
    -------
    float in [0.0, 1.0]. Return 0.0 for an empty sequence.
    """
    if not episodes:
        return 0.0
    return sum(not bool(e["crashed"]) for e in episodes) / len(episodes)


def mean_survival_steps(episodes):
    """Return the mean of the "length" field across all episodes.

    Returns
    -------
    float. Return 0.0 for an empty sequence.
    """
    if not episodes:
        return 0.0
    return sum(float(e["length"]) for e in episodes) / len(episodes)


def summarize_rollout(episodes):
    """Aggregate per-episode records into one summary dict. Provided -- not a TODO.
    This is what results.json is built from; it will work once the three functions
    above are implemented correctly."""
    n = len(episodes)
    mean_reward = (sum(float(e["reward"]) for e in episodes) / n) if n else 0.0
    return {
        "n_episodes": n,
        "mean_reward": mean_reward,
        "collision_rate": collision_rate(episodes),
        "success_rate": success_rate(episodes),
        "mean_survival_steps": mean_survival_steps(episodes),
    }

# === AUTOGRADER_EXTRACT_END: diagnostics ===


#### Self-check — run this until it prints `ALL CHECKS PASSED`

These are the same categories of test the autograder's **hidden** suite uses on different fixtures
— passing all of these is a strong (not guaranteed-identical) signal you will pass that component
too.


In [4]:
def _check(name, condition):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {name}")
    return condition


def run_diagnostics_self_check():
    import math

    empty = []
    mixed = [
        {"reward": 10.0, "length": 30, "crashed": False},
        {"reward": -5.0, "length": 12, "crashed": True},
        {"reward": 8.0, "length": 30, "crashed": False},
        {"reward": -2.0, "length": 5, "crashed": True},
    ]
    all_safe = [
        {"reward": 9.0, "length": 30, "crashed": False},
        {"reward": 11.0, "length": 25, "crashed": False},
    ]
    all_crashed = [
        {"reward": -1.0, "length": 3, "crashed": True},
        {"reward": -1.0, "length": 7, "crashed": True},
        {"reward": -1.0, "length": 2, "crashed": True},
    ]

    results = []
    results.append(_check("collision_rate([]) == 0.0", collision_rate(empty) == 0.0))
    results.append(_check("collision_rate(mixed) == 0.5", math.isclose(collision_rate(mixed), 0.5)))
    results.append(_check("collision_rate(all_safe) == 0.0", collision_rate(all_safe) == 0.0))
    results.append(_check("collision_rate(all_crashed) == 1.0", collision_rate(all_crashed) == 1.0))

    results.append(_check("success_rate([]) == 0.0", success_rate(empty) == 0.0))
    results.append(_check("success_rate(mixed) == 0.5", math.isclose(success_rate(mixed), 0.5)))
    results.append(_check("success_rate(all_safe) == 1.0", success_rate(all_safe) == 1.0))
    results.append(_check("success_rate(all_crashed) == 0.0", success_rate(all_crashed) == 0.0))
    results.append(_check(
        "success_rate + collision_rate == 1.0 (mixed/all_safe/all_crashed)",
        all(math.isclose(success_rate(e) + collision_rate(e), 1.0) for e in (mixed, all_safe, all_crashed)),
    ))

    results.append(_check("mean_survival_steps([]) == 0.0", mean_survival_steps(empty) == 0.0))
    results.append(_check("mean_survival_steps(mixed) == 19.25", math.isclose(mean_survival_steps(mixed), 19.25)))
    results.append(_check(
        "mean_survival_steps(constant length=10) == 10.0",
        math.isclose(mean_survival_steps([{"reward": 0.0, "length": 10, "crashed": False}] * 5), 10.0),
    ))

    summary = summarize_rollout(mixed)
    expected_keys = {"n_episodes", "mean_reward", "collision_rate", "success_rate", "mean_survival_steps"}
    results.append(_check("summarize_rollout keys present", expected_keys <= summary.keys()))
    results.append(_check("summarize_rollout values correct", (
        summary["n_episodes"] == 4
        and math.isclose(summary["mean_reward"], (10.0 - 5.0 + 8.0 - 2.0) / 4)
        and math.isclose(summary["collision_rate"], 0.5)
        and math.isclose(summary["success_rate"], 0.5)
        and math.isclose(summary["mean_survival_steps"], 19.25)
    )))
    results.append(_check("summarize_rollout([]) all zero", all(
        v == 0 for k, v in summarize_rollout(empty).items() if k != "n_episodes"
    )))

    if all(results):
        print(f"\nALL CHECKS PASSED ({sum(results)}/{len(results)})")
    else:
        print(f"\n{sum(results)}/{len(results)} checks passed -- fix the failing ones above before continuing.")
    return all(results)


assert run_diagnostics_self_check(), "Fix diagnostics.py logic above before proceeding -- do not continue to Step 2 until this passes."


  [PASS] collision_rate([]) == 0.0
  [PASS] collision_rate(mixed) == 0.5
  [PASS] collision_rate(all_safe) == 0.0
  [PASS] collision_rate(all_crashed) == 1.0
  [PASS] success_rate([]) == 0.0
  [PASS] success_rate(mixed) == 0.5
  [PASS] success_rate(all_safe) == 1.0
  [PASS] success_rate(all_crashed) == 0.0
  [PASS] success_rate + collision_rate == 1.0 (mixed/all_safe/all_crashed)
  [PASS] mean_survival_steps([]) == 0.0
  [PASS] mean_survival_steps(mixed) == 19.25
  [PASS] mean_survival_steps(constant length=10) == 10.0
  [PASS] summarize_rollout keys present
  [PASS] summarize_rollout values correct
  [PASS] summarize_rollout([]) all zero

ALL CHECKS PASSED (15/15)


### `shaping` — potential-based reward shaping (OPTIONAL — Optional Task 2 only, ungraded)

Lecture reference: *"Reward Shaping: Densifying the Signal"* (slide 56) and *"The Anatomy of the
Shaping Function"* (slide 57): `F(s, s') = gamma * Phi(s') - Phi(s)`.

Not required for the mandatory task. Skip to **Mandatory Task** below and come back to this later
if you want to attempt Optional Task 2.


In [5]:
def potential(state_summary):
    """Return the potential Phi(s) of a state summary dict, e.g.
    {"distance_to_goal": 12.4, "speed": 3.1}. High near the goal, low far from it."""
    distance = float(state_summary["distance_to_goal"])
    speed = float(state_summary.get("speed", 0.0))
    # Higher potential means closer to the goal; a small speed penalty
    # discourages arriving with excessive residual motion.
    return -distance - 0.1 * speed


def shape_reward(state_summary, next_state_summary, gamma, base_reward):
    """Return base_reward + F(s, s') = base_reward + gamma * potential(next) - potential(current).
    `gamma` MUST match the discount factor used by your training algorithm."""
    return float(
        base_reward
        + gamma * potential(next_state_summary)
        - potential(state_summary)
    )


def parking_state_summary(obs):
    """Provided helper for Optional Task 2 -- builds a state-summary dict from a raw
    parking-v0 (GoalEnv-style) observation: {"observation", "achieved_goal", "desired_goal"},
    each a 6-D vector [x, y, vx, vy, cos_h, sin_h]."""
    achieved = np.asarray(obs["achieved_goal"], dtype=float)
    desired = np.asarray(obs["desired_goal"], dtype=float)
    distance_to_goal = float(np.linalg.norm(achieved[:2] - desired[:2]))
    speed = float(np.linalg.norm(achieved[2:4]))
    return {"distance_to_goal": distance_to_goal, "speed": speed}


def run_shaping_self_check():
    import math
    near = {"distance_to_goal": 1.0, "speed": 0.5}
    far = {"distance_to_goal": 10.0, "speed": 0.5}
    goal = {"distance_to_goal": 0.0, "speed": 0.0}

    results = []
    results.append(_check("potential(near) > potential(far)", potential(near) > potential(far)))
    results.append(_check("potential(goal) is the maximum", potential(goal) >= potential(near) and potential(goal) >= potential(far)))
    gamma = 0.9
    expected = 0.0 + gamma * potential(near) - potential(far)
    results.append(_check("shape_reward matches F(s,s') formula", math.isclose(shape_reward(far, near, gamma, 0.0), expected)))
    results.append(_check(
        "shape_reward adds to base_reward",
        math.isclose(shape_reward(far, near, gamma, 5.0) - shape_reward(far, near, gamma, 0.0), 5.0),
    ))
    results.append(_check(
        "moving toward goal is rewarded more than moving away",
        shape_reward(far, near, gamma, 0.0) > shape_reward(near, far, gamma, 0.0),
    ))
    if all(results):
        print(f"\nALL CHECKS PASSED ({sum(results)}/{len(results)})")
    else:
        print(f"\n{sum(results)}/{len(results)} checks passed.")
    return all(results)

# Uncomment when you attempt Optional Task 2:
# assert run_shaping_self_check()


### `envs` — environment construction (provided)

`MANDATORY_ENV_CONFIG` matches the autograder's own grading configuration — **do not change it**
for the mandatory task (you are free to change it for Optional Tasks 1 and 3).


In [6]:
MANDATORY_ENV_ID = "highway-fast-v0"

MANDATORY_ENV_CONFIG = {
    "lanes_count": 4,
    "vehicles_count": 20,
    "duration": 30,
    "policy_frequency": 1,
    "collision_reward": -1,
    "right_lane_reward": 0.1,
    "high_speed_reward": 0.4,
    "reward_speed_range": [20, 30],
    "normalize_reward": True,
    "observation": {"type": "Kinematics"},
    "action": {"type": "DiscreteMetaAction"},
}


def make_env(env_id=MANDATORY_ENV_ID, config=None, seed=None, render_mode=None):
    """Construct a highway-env Gymnasium environment. `config` overrides are merged on
    top of MANDATORY_ENV_CONFIG. Pass render_mode="rgb_array" to enable env.render() frame
    capture for GIF/MP4 recording."""
    cfg = dict(MANDATORY_ENV_CONFIG)
    if config:
        cfg.update(config)
    env = gym.make(env_id, config=cfg, render_mode=render_mode)
    if seed is not None:
        env.reset(seed=seed)
    return env


### `train` — DQN / PPO training wrappers (provided)

Your job is to **configure** `dqn_kwargs` in Step 3 below (pick `gamma`, `buffer_size`,
`target_update_interval`, ...) and justify your choices — not to re-implement DQN from scratch.


In [7]:
from copy import deepcopy
from stable_baselines3.common.monitor import Monitor

MANDATORY_TIMESTEP_BUDGET = 20_000  # MUST remain <= this for the mandatory task.

DEFAULT_DQN_KWARGS = dict(
    policy="MlpPolicy",
    policy_kwargs=dict(net_arch=[128, 128]),
    learning_rate=5e-4,
    buffer_size=15_000,
    learning_starts=500,
    batch_size=64,
    gamma=0.8,
    train_freq=1,
    gradient_steps=1,
    target_update_interval=1000,
    exploration_fraction=0.15,
    exploration_final_eps=0.05,
    verbose=0,
)

def train_dqn(
    env,
    total_timesteps=MANDATORY_TIMESTEP_BUDGET,
    seed=0,
    dqn_kwargs=None,
    monitor_path="artifacts/monitor.csv",
    callback=None,
    progress_bar=False,
):
    """Train DQN and return the fitted SB3 model."""
    if total_timesteps <= 0 or total_timesteps > MANDATORY_TIMESTEP_BUDGET:
        raise ValueError(
            f"Mandatory training must use 1..{MANDATORY_TIMESTEP_BUDGET} timesteps; "
            f"got {total_timesteps}."
        )

    if monitor_path:
        out_dir = os.path.dirname(monitor_path)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
        env = Monitor(env, filename=monitor_path)

    kwargs = deepcopy(DEFAULT_DQN_KWARGS)
    if dqn_kwargs:
        kwargs.update(dqn_kwargs)

    model = DQN(
        env=env,
        seed=seed,
        device="cpu",
        **kwargs,
    )
    model.learn(
        total_timesteps=total_timesteps,
        callback=callback,
        progress_bar=progress_bar,
    )
    return model


# Optional Task 1 (ungraded): PPO on continuous control.
DEFAULT_PPO_KWARGS = dict(
    policy="MlpPolicy",
    n_steps=512,
    batch_size=64,
    gae_lambda=0.95,
    gamma=0.8,
    n_epochs=10,
    ent_coef=0.01,
    clip_range=0.2,
    learning_rate=3e-4,
    verbose=0,
)

def train_ppo(
    env,
    total_timesteps=40_000,
    seed=0,
    ppo_kwargs=None,
    monitor_path="artifacts/monitor_ppo.csv",
    callback=None,
):
    if monitor_path:
        out_dir = os.path.dirname(monitor_path)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
        env = Monitor(env, filename=monitor_path)
    kwargs = deepcopy(DEFAULT_PPO_KWARGS)
    if ppo_kwargs:
        kwargs.update(ppo_kwargs)
    model = PPO(env=env, seed=seed, device="cpu", **kwargs)
    model.learn(total_timesteps=total_timesteps, callback=callback)
    return model

### `evaluate` — deterministic rollout

`rollout` with no explicit `seeds` uses seeds `0..n-1` — the same convention the autograder uses, so
your local numbers are directly comparable to your grade (the autograder still re-evaluates your
*saved model file* independently, not this in-memory object).


In [8]:
def rollout(model, env, n_episodes=100, seeds=None, deterministic=True):
    """Run `model` in `env` for n_episodes and return a list of episode records:
    {"reward": float, "length": int, "crashed": bool}."""
    records = []
    for i in range(n_episodes):
        seed = seeds[i] if seeds is not None else i
        obs, info = env.reset(seed=seed)
        terminated = truncated = False
        total_reward = 0.0
        steps = 0
        crashed = bool(info.get("crashed", False))
        while not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += float(reward)
            steps += 1
            crashed = bool(info.get("crashed", crashed))
        records.append({"reward": total_reward, "length": steps, "crashed": crashed})
    return records


### `record` — GIF / MP4 rollout recording

In [9]:
def save_gif(frames, path, fps=10):
    if not frames:
        raise ValueError("save_gif received an empty frame sequence")
    out_dir = os.path.dirname(path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    images = [Image.fromarray(np.asarray(f).astype("uint8")) for f in frames]
    duration_ms = max(1, int(1000 / fps))
    images[0].save(path, save_all=True, append_images=images[1:], duration=duration_ms, loop=0, optimize=True)
    return path


def save_mp4(frames, path, fps=10):
    if not frames:
        raise ValueError("save_mp4 received an empty frame sequence")
    import cv2
    out_dir = os.path.dirname(path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    first = np.asarray(frames[0])
    h, w = first.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(path, fourcc, fps, (w, h))
    if not writer.isOpened():
        path = os.path.splitext(path)[0] + ".avi"
        fourcc = cv2.VideoWriter_fourcc(*"MJPG")
        writer = cv2.VideoWriter(path, fourcc, fps, (w, h))
    for frame in frames:
        rgb = np.asarray(frame).astype("uint8")
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        writer.write(bgr)
    writer.release()
    return path


def record_rollout(model, env, n_episodes=1, deterministic=True):
    frames = []
    for _ in range(n_episodes):
        obs, info = env.reset()
        frames.append(env.render())
        done = truncated = False
        while not (done or truncated):
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, reward, done, truncated, info = env.step(action)
            frames.append(env.render())
    return frames


### `plotting` — training-curve plot

Implements the lecture's "Golden Rule of Diagnostics" (slide 51): plot reward **alongside** a
secondary signal, never reward alone.


In [10]:
def _rolling_mean(values, window):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return values
    window = max(1, min(window, len(values)))
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="valid")


def plot_training_curve(episode_rewards, secondary=None, secondary_label="Success rate",
                         save_path="artifacts/training_curve.png", window=20,
                         title="Training curve: reward vs. secondary signal"):
    out_dir = os.path.dirname(save_path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    fig, ax1 = plt.subplots(figsize=(8, 4.5))
    rewards = np.asarray(episode_rewards, dtype=float)
    ax1.plot(rewards, color="tab:blue", alpha=0.25, linewidth=1, label="Episode reward (raw)")
    if len(rewards) > 1:
        smoothed = _rolling_mean(rewards, window)
        x_smooth = np.arange(len(smoothed)) + (len(rewards) - len(smoothed))
        ax1.plot(x_smooth, smoothed, color="tab:blue", linewidth=2, label=f"Episode reward ({window}-ep rolling mean)")
    ax1.set_xlabel("Episode")
    ax1.set_ylabel("Episode reward", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")
    lines, labels = ax1.get_legend_handles_labels()

    if secondary is not None and len(secondary) > 0:
        ax2 = ax1.twinx()
        secondary_arr = np.asarray(secondary, dtype=float)
        x_secondary = np.linspace(0, max(len(rewards) - 1, 1), num=len(secondary_arr))
        ax2.plot(x_secondary, secondary_arr, color="tab:red", linewidth=2, linestyle="--", label=secondary_label)
        ax2.set_ylabel(secondary_label, color="tab:red")
        ax2.tick_params(axis="y", labelcolor="tab:red")
        l2, lab2 = ax2.get_legend_handles_labels()
        lines += l2
        labels += lab2

    ax1.set_title(title)
    ax1.legend(lines, labels, loc="lower right", fontsize=8)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    return save_path


### `results` — results.json schema + I/O

The autograder cross-checks the numbers you self-report here against its own re-evaluation of your
submitted model.


In [11]:
REQUIRED_SUMMARY_KEYS = {"n_episodes", "mean_reward", "collision_rate", "success_rate", "mean_survival_steps"}
REQUIRED_META_KEYS = {"algo", "env_id", "total_timesteps", "seed"}


def write_results(path, summary, **meta):
    payload = dict(summary)
    payload.update(meta)
    missing_summary = REQUIRED_SUMMARY_KEYS - set(summary.keys())
    missing_meta = REQUIRED_META_KEYS - set(meta.keys())
    if missing_summary or missing_meta:
        raise ValueError(f"results.json missing required keys: summary={sorted(missing_summary)} meta={sorted(missing_meta)}")
    out_dir = os.path.dirname(path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    with open(path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)
    return path


def read_results(path):
    with open(path) as f:
        payload = json.load(f)
    missing = (REQUIRED_SUMMARY_KEYS | REQUIRED_META_KEYS) - set(payload.keys())
    if missing:
        raise ValueError(f"{path} missing required keys: {sorted(missing)}")
    return payload


### `callbacks` — in-training diagnostics

Periodically pauses training to roll out a few deterministic evaluation episodes and record your
`summarize_rollout(...)` output, so you can plot a secondary signal alongside reward (Step 5).


In [12]:
from stable_baselines3.common.callbacks import BaseCallback

class DiagnosticsCallback(BaseCallback):
    """Periodic deterministic evaluation for a secondary training signal."""
    def __init__(self, eval_env, eval_every_steps=2000, n_eval_episodes=5, verbose=0):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_every_steps = int(eval_every_steps)
        self.n_eval_episodes = int(n_eval_episodes)
        self.history = []
        self._last_eval_step = 0

    def _on_step(self) -> bool:
        if self.num_timesteps - self._last_eval_step >= self.eval_every_steps:
            self._last_eval_step = self.num_timesteps
            episodes = rollout(
                self.model,
                self.eval_env,
                n_episodes=self.n_eval_episodes,
                seeds=list(range(1000, 1000 + self.n_eval_episodes)),
                deterministic=True,
            )
            summary = summarize_rollout(episodes)
            summary["timestep"] = int(self.num_timesteps)
            self.history.append(summary)
            if self.verbose:
                print(f"[DiagnosticsCallback] step={self.num_timesteps}: {summary}")
        return True

---
## MANDATORY TASK

### Step 1 — recap: diagnostics self-check already passed above ✅

If `run_diagnostics_self_check()` printed `ALL CHECKS PASSED` above, you're ready to continue.
(If not, scroll back up and fix `diagnostics` before going further — everything below depends on it.)

### Step 2 — Build the environment


**Reproducibility:** the mandatory training run uses seed `0`; evaluation uses seeds `0..19`.

In [13]:
train_env = make_env(MANDATORY_ENV_ID)  # no rendering while training (faster)
print("Observation space:", train_env.observation_space)
print("Action space:", train_env.action_space)  # DiscreteMetaAction: 5 actions (slide 15)


Observation space: Box(-inf, inf, (5, 5), float32)
Action space: Discrete(5)


### Step 3 — Configure and train DQN

Choose and justify at least three hyperparameters, each tied to a lecture concept:

| Hyperparameter | Choice | Lecture concept |
| --- | ---: | --- |
| `gamma` | `0.8` | Discounting / agent horizon |
| `buffer_size` | `15,000` | Experience Replay |
| `target_update_interval` | `1,000` | Target Networks |
| `exploration_fraction` | `0.15` | Exploration vs. Exploitation |
| `exploration_final_eps` | `0.05` | Residual exploration |

The mandatory budget is **20,000 environment timesteps maximum**. The code below uses exactly
20,000 steps.

`gamma=0.8` gives a relatively short effective horizon for this fast highway task. A
`15,000`-transition replay buffer provides varied samples and reduces temporal correlation.
A target-network update interval of `1,000` steps keeps the bootstrap target fixed for a
meaningful interval while still updating several times during training. Exploration decays
over the first 15% of training and finishes at 5% random action probability.

The network is kept at `[128, 128]` so the mandatory 20,000-step CPU run remains practical.

In [14]:
import warnings
import logging

warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("stable_baselines3").setLevel(logging.ERROR)

class CleanProgressCallback(BaseCallback):
    """Compact progress output without SB3/Rich live-progress conflicts."""
    def __init__(self, total_steps, check_freq=1000):
        super().__init__(verbose=0)
        self.total_steps = int(total_steps)
        self.check_freq = int(check_freq)

    def _on_step(self) -> bool:
        if self.n_calls % self.check_freq == 0 or self.n_calls >= self.total_steps:
            pct = 100.0 * self.n_calls / self.total_steps
            print(f"Training Progress: {self.n_calls}/{self.total_steps} steps ({pct:.1f}%)")
        return True

# Mandatory hyperparameters and their lecture rationale:
# gamma=0.8: finite planning horizon for the fast highway task.
# buffer_size=15,000: replay memory reduces temporal correlation.
# target_update_interval=1,000: frozen target network stabilises bootstrapping.
# exploration_fraction=0.15: gradual exploration -> exploitation transition.
dqn_kwargs = dict(
    gamma=0.8,
    buffer_size=15_000,
    target_update_interval=1000,
    exploration_fraction=0.15,
    exploration_final_eps=0.05,
)

TOTAL_STEPS = MANDATORY_TIMESTEP_BUDGET
SEED = 0

os.makedirs("artifacts", exist_ok=True)
monitor_path = "artifacts/monitor.csv"
if os.path.exists(monitor_path):
    os.remove(monitor_path)

train_env = make_env(MANDATORY_ENV_ID, seed=SEED)
diag_env = make_env(MANDATORY_ENV_ID, seed=1000)

diag_callback = DiagnosticsCallback(
    eval_env=diag_env,
    eval_every_steps=1000,
    n_eval_episodes=5,
    verbose=0,
)
progress_callback = CleanProgressCallback(
    total_steps=TOTAL_STEPS,
    check_freq=1000,
)
callbacks = CallbackList([diag_callback, progress_callback])

model = train_dqn(
    train_env,
    total_timesteps=TOTAL_STEPS,
    seed=SEED,
    dqn_kwargs=dqn_kwargs,
    monitor_path=monitor_path,
    callback=callbacks,
    progress_bar=False,
)

assert model.num_timesteps == TOTAL_STEPS, (
    f"Unexpected training length: {model.num_timesteps}; expected {TOTAL_STEPS}"
)
print(f"Training complete: {model.num_timesteps} timesteps.")

Training Progress: 1000/20000 steps (5.0%)
Training Progress: 2000/20000 steps (10.0%)
Training Progress: 3000/20000 steps (15.0%)
Training Progress: 4000/20000 steps (20.0%)
Training Progress: 5000/20000 steps (25.0%)
Training Progress: 6000/20000 steps (30.0%)
Training Progress: 7000/20000 steps (35.0%)
Training Progress: 8000/20000 steps (40.0%)
Training Progress: 9000/20000 steps (45.0%)
Training Progress: 10000/20000 steps (50.0%)
Training Progress: 11000/20000 steps (55.0%)
Training Progress: 12000/20000 steps (60.0%)
Training Progress: 13000/20000 steps (65.0%)
Training Progress: 14000/20000 steps (70.0%)
Training Progress: 15000/20000 steps (75.0%)
Training Progress: 16000/20000 steps (80.0%)
Training Progress: 17000/20000 steps (85.0%)
Training Progress: 18000/20000 steps (90.0%)
Training Progress: 19000/20000 steps (95.0%)
Training Progress: 20000/20000 steps (100.0%)
Training complete: 20000 timesteps.


### Step 4 — Save your trained model

This exact file is what the autograder loads — **do not rename it or change the save path**.


In [15]:
os.makedirs("artifacts", exist_ok=True)
MODEL_PATH = "artifacts/dqn_highway.zip"
model.save(MODEL_PATH)

assert os.path.isfile(MODEL_PATH), "Model file was not created."
print(f"Saved to {MODEL_PATH}")

Saved to artifacts/dqn_highway.zip


> Please ensure that you commit `artifacts/dqn_highway.zip` in your GitHub repository (otherwise, it won't be part of your Gradescope submission)

### Step 5 — Evaluate, plot, record, and write `results.json`


In [16]:
eval_env = make_env(MANDATORY_ENV_ID, render_mode="rgb_array")
evaluation_seeds = list(range(20))
episodes = rollout(
    model,
    eval_env,
    n_episodes=20,
    seeds=evaluation_seeds,
    deterministic=True,
)
summary = summarize_rollout(episodes)

print("Evaluation summary:")
for key, value in summary.items():
    print(f"  {key}: {value}")

Evaluation summary:
  n_episodes: 20
  mean_reward: 21.97934895606831
  collision_rate: 0.5
  success_rate: 0.5
  mean_survival_steps: 25.05


In [17]:
monitor_df = pd.read_csv("artifacts/monitor.csv", skiprows=1)

secondary = [h["success_rate"] for h in diag_callback.history]
plot_training_curve(
    monitor_df["r"].tolist(),
    secondary=secondary,
    secondary_label="Success rate (5-episode eval)",
    save_path="artifacts/training_curve.png",
)

assert os.path.isfile("artifacts/training_curve.png")
print("Saved artifacts/training_curve.png")

Saved artifacts/training_curve.png


In [18]:
frames = record_rollout(model, eval_env, n_episodes=1, deterministic=True)
assert len(frames) > 0, "No frames were recorded."

gif_path = save_gif(frames, "artifacts/rollout.gif", fps=10)
mp4_path = save_mp4(frames, "artifacts/rollout.mp4", fps=10)

print(f"Recorded {len(frames)} frames.")
print(f"GIF: {gif_path}")
print(f"Video: {mp4_path}")

Recorded 19 frames.
GIF: artifacts/rollout.gif
Video: artifacts/rollout.mp4


In [19]:
RESULTS_PATH = "artifacts/results.json"

write_results(
    RESULTS_PATH,
    summary,
    algo="dqn",
    env_id=MANDATORY_ENV_ID,
    total_timesteps=TOTAL_STEPS,
    seed=SEED,
)

saved_results = read_results(RESULTS_PATH)

assert saved_results["n_episodes"] == 20
assert saved_results["env_id"] == MANDATORY_ENV_ID
assert saved_results["total_timesteps"] <= MANDATORY_TIMESTEP_BUDGET
assert abs(saved_results["collision_rate"] + saved_results["success_rate"] - 1.0) < 1e-12

print("Wrote artifacts/results.json")
print(json.dumps(saved_results, indent=2))

Wrote artifacts/results.json
{
  "algo": "dqn",
  "collision_rate": 0.5,
  "env_id": "highway-fast-v0",
  "mean_reward": 21.97934895606831,
  "mean_survival_steps": 25.05,
  "n_episodes": 20,
  "seed": 0,
  "success_rate": 0.5,
  "total_timesteps": 20000
}


In [20]:
# Final mandatory-artifact verification
required_artifacts = [
    "artifacts/dqn_highway.zip",
    "artifacts/training_curve.png",
    "artifacts/rollout.gif",
    "artifacts/results.json",
]
missing = [p for p in required_artifacts if not os.path.isfile(p)]

if missing:
    raise FileNotFoundError(f"Missing mandatory artifacts: {missing}")

assert model.num_timesteps <= MANDATORY_TIMESTEP_BUDGET
assert os.path.getsize("artifacts/dqn_highway.zip") > 0

print("ALL MANDATORY ARTIFACTS PRESENT")
for path in required_artifacts:
    print(f"  ✓ {path}")

# Close environments after all rendering/recording work is complete.
try:
    train_env.close()
    diag_env.close()
    eval_env.close()
except Exception:
    pass


ALL MANDATORY ARTIFACTS PRESENT
  ✓ artifacts/dqn_highway.zip
  ✓ artifacts/training_curve.png
  ✓ artifacts/rollout.gif
  ✓ artifacts/results.json


### Mandatory task submission checklist

- [ ] `run_diagnostics_self_check()` prints `ALL CHECKS PASSED`
- [ ] `artifacts/dqn_highway.zip` saved
- [ ] `artifacts/training_curve.png` saved (reward **and** secondary signal visible)
- [ ] `artifacts/rollout.gif` and/or `.mp4` saved
- [ ] `artifacts/results.json` saved
- [ ] Notebook committed with the `AUTOGRADER_EXTRACT_START/END` markers around the
      diagnostics cell left intact
- [ ] PR opened, reviewed and merged
- [ ] Notebook/artifacts submitted to Gradescope

**Important:** the trained model and numerical results must be generated by running the notebook
in an environment with the required packages. Do not fabricate evaluation numbers.